# kaiming-uniform-sf-init — faded example 3: Conv2d SF init — fill in the fan_in formula for spatial kernels

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-sf-init`. Running the beacon reports progress on the `Init: Kaiming uniform SF init` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform SF init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-sf-init`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-sf-init"
DD_SUBTOPIC = "Init: Kaiming uniform SF init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For a Conv2d weight of shape `(out_ch, in_ch, kH, kW)`, the `fan_in` is `in_ch * kH * kW` because each output element is the sum of products over all input channels AND all kernel positions. The scale factor `sf = 1/sqrt(fan_in)` must use this expanded fan-in.

## Faded exercise 3

The `kaiming_uniform_sf_conv2d` function body is provided but the `fan_in` computation is missing. Fill it in using the Conv2d-specific formula.

**Fill in:** Compute fan_in as in_channels times kernel_h times kernel_w.

In [ ]:
import torch as t

def kaiming_uniform_sf_conv2d(out_channels: int, in_channels: int, kernel_h: int, kernel_w: int, generator: t.Generator) -> t.Tensor:
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf


def _test():
    import torch as t

    g = t.Generator()
    g.manual_seed(2)
    w = kaiming_uniform_sf_conv2d(16, 4, 3, 3, g)
    fan_in = 4 * 3 * 3  # 36
    sf = fan_in ** -0.5
    assert w.shape == (16, 4, 3, 3)
    assert w.min().item() >= -sf - 1e-6
    assert w.max().item() <=  sf + 1e-6
    # compare with 1x1 version: 1x1 sf should be 3x larger
    g.manual_seed(2)
    w_1x1 = kaiming_uniform_sf_conv2d(16, 4, 1, 1, g)
    sf_1x1 = (4 * 1 * 1) ** -0.5
    assert abs(sf_1x1 / sf - 3.0) < 0.01


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def kaiming_uniform_sf_conv2d(out_channels: int, in_channels: int, kernel_h: int, kernel_w: int, generator: t.Generator) -> t.Tensor:
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf
```
</details>